In [ ]:
import rasterio
import leafmap.foliumap as leafmap
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from utils.raster import load_tif
from utils.segmentation import build_segments, segment_attributes, segments_to_gdf
from utils.attributes import (
    compute_geometry_attributes,
    compute_image_attributes
)

# 1. Load the TIFF image

We first load the TIFF and optionally downsample it to make segmentation faster.

This returns:

- `img` → the image as a NumPy array
- `crs` → the coordinate reference system
- `transform` → the spatial transform linking pixels to map coordinates

In [ ]:
# Set tif path
original_path = "./data/vlasakkers.tif"

# Set downscaled preview path
preview_path = "./data/vlasakkers_preview.tif"

# Load
img, crs, transform = load_tif(original_path, resize=4)

# 2. Build segments

We use Felzenszwalb segmentation to group similar neighboring pixels into meaningful regions.

We then compute attributes per segment and convert them into a GeoDataFrame for further spatial analysis.

In [ ]:
# Build segments
segments = build_segments(img, scale=250, sigma=0.5, min_size=150)

In [ ]:
# Add segment attributes
attrs = segment_attributes(img, segments)

In [ ]:
# Convert to GeoDataFrame
segments_gdf = segments_to_gdf(segments, transform, crs)

# Add attributes to the GeoDataFrame
segments_gdf = segments_gdf.merge(attrs, on="segment_id", how="left")

In [ ]:
segments_gdf.head()

# 3. Enrich segments

We enrich each segment with additional spatial and contextual attributes.

This lets us move from image-based regions to analysis-ready terrain units.

In [ ]:
segments_gdf = compute_geometry_attributes(segments_gdf)
segments_gdf = compute_image_attributes(segments_gdf)
segments_gdf.head()

In [ ]:
cols = [
    "segment_id",
    "area",
    "compactness",
    "brightness",
    "texture_score",
    "geometry"
]

segments_view = segments_gdf[cols].copy()

In [ ]:
m = leafmap.Map()
m.add_raster(str(preview_path), layer_name="Image")

m.add_gdf(
    segments_view,
    layer_name="Segments",
    style={
        "color": "yellow",
        "weight": 1,
        "fillOpacity": 0,
    },
)

m.zoom_to_bounds(str(preview_path))
m

